# Train PyTorch record-level autoencoders

Trains one independent PyTorch autoencoder per lakehouse layer from `report_aiops_record_features`, which is produced by `01_aiops_build_record_features`.

The model learns the normal record-level shape of each layer separately.

In [ ]:
%pip install torch --quiet

In [ ]:
from datetime import date
import base64
from io import BytesIO
import json

from pyspark.sql import functions as F
from pyspark.sql.window import Window

try:
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
except ImportError as exc:
    raise ImportError("Use a Databricks ML Runtime or install torch before running this notebook.") from exc

dbutils.widgets.text("catalog_name", "hant-catalog")
dbutils.widgets.text("schema_name", "hsl")
dbutils.widgets.text("feature_table", "report_aiops_record_features")
dbutils.widgets.text("model_version", "")
dbutils.widgets.text("epochs", "100")
dbutils.widgets.text("hidden_dim", "16")
dbutils.widgets.text("anomaly_percentile", "0.99")
dbutils.widgets.text("min_train_records", "100")
dbutils.widgets.text("max_train_records_per_layer", "20000")
dbutils.widgets.text("train_sample_seed", "42")
dbutils.widgets.text("train_start_date", "")
dbutils.widgets.text("train_end_date", "")
dbutils.widgets.dropdown("train_only_rule_clean_records", "true", ["true", "false"])
dbutils.widgets.dropdown("include_rule_flag_features", "true", ["true", "false"])
dbutils.widgets.text("storage_account", "streanmingdatasta")
dbutils.widgets.text("lakehouse_container", "lakehouse")
dbutils.widgets.text("model_base_path", "")

CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
FEATURE_TABLE = dbutils.widgets.get("feature_table").strip() or "report_aiops_record_features"
MODEL_VERSION = dbutils.widgets.get("model_version").strip() or date.today().isoformat()
EPOCHS = int(dbutils.widgets.get("epochs"))
HIDDEN_DIM = int(dbutils.widgets.get("hidden_dim"))
ANOMALY_PERCENTILE = float(dbutils.widgets.get("anomaly_percentile"))
MIN_TRAIN_RECORDS = int(dbutils.widgets.get("min_train_records"))
MAX_TRAIN_RECORDS_PER_LAYER = int(dbutils.widgets.get("max_train_records_per_layer"))
TRAIN_SAMPLE_SEED = int(dbutils.widgets.get("train_sample_seed"))
TRAIN_START_DATE = dbutils.widgets.get("train_start_date").strip()
TRAIN_END_DATE = dbutils.widgets.get("train_end_date").strip()
TRAIN_ONLY_RULE_CLEAN_RECORDS = dbutils.widgets.get("train_only_rule_clean_records").lower() == "true"
INCLUDE_RULE_FLAG_FEATURES = dbutils.widgets.get("include_rule_flag_features").lower() == "true"
STORAGE_ACCOUNT = dbutils.widgets.get("storage_account")
LAKEHOUSE_CONTAINER = dbutils.widgets.get("lakehouse_container")
MODEL_BASE_PATH_WIDGET = dbutils.widgets.get("model_base_path").strip()
MODEL_BASE_PATH = MODEL_BASE_PATH_WIDGET or f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/external/hant-catalog/aiops/models"

MODEL_ARTIFACTS_TABLE = "report_aiops_model_artifacts"
TRAINING_SUMMARY_TABLE = "report_aiops_training_summary"


def qname(table_name: str) -> str:
    return f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{table_name}"


print(json.dumps({
    "feature_table": qname(FEATURE_TABLE),
    "model_version": MODEL_VERSION,
    "epochs": EPOCHS,
    "hidden_dim": HIDDEN_DIM,
    "anomaly_percentile": ANOMALY_PERCENTILE,
    "min_train_records": MIN_TRAIN_RECORDS,
    "max_train_records_per_layer": MAX_TRAIN_RECORDS_PER_LAYER,
    "train_sample_seed": TRAIN_SAMPLE_SEED,
    "train_start_date": TRAIN_START_DATE or None,
    "train_end_date": TRAIN_END_DATE or None,
    "train_only_rule_clean_records": TRAIN_ONLY_RULE_CLEAN_RECORDS,
    "include_rule_flag_features": INCLUDE_RULE_FLAG_FEATURES,
    "model_base_path": MODEL_BASE_PATH,
}, indent=2))

In [ ]:
features_df = spark.table(qname(FEATURE_TABLE))
required_cols = {"record_id", "layer", "event_ts", "feature_date"}
missing_required = required_cols - set(features_df.columns)
if missing_required:
    raise ValueError(f"Feature table {qname(FEATURE_TABLE)} is missing required columns: {sorted(missing_required)}")

metadata_cols = {
    "layer", "record_id", "business_key", "event_ts", "feature_date",
    "source_table", "rule_metadata_json",
}
summary_label_cols = {
    "critical_rule_fail_count", "high_rule_fail_count", "medium_rule_fail_count", "low_rule_fail_count",
    "total_rule_fail_count", "has_rule_failure_flag", "max_rule_severity_rank",
}
numeric_spark_types = {
    "byte", "short", "int", "bigint", "long", "float", "double", "decimal",
}

candidate_feature_cols = []
for col_name, dtype in features_df.dtypes:
    dtype_base = dtype.split("(")[0].lower()
    if col_name in metadata_cols:
        continue
    if col_name in summary_label_cols:
        continue
    if not INCLUDE_RULE_FLAG_FEATURES and col_name.startswith("rule_") and col_name.endswith("_flag"):
        continue
    if dtype_base in numeric_spark_types:
        candidate_feature_cols.append(col_name)

if not candidate_feature_cols:
    raise ValueError("No numeric candidate feature columns found in the record feature table.")

clean_filter_cols = [
    "has_rule_failure_flag",
    "total_rule_fail_count",
    "critical_rule_fail_count",
    "high_rule_fail_count",
    "rule_medium_producer_ts_bad_format_flag",
]
selected_cols = list(dict.fromkeys([
    "layer", "record_id", "business_key", "event_ts", "feature_date",
    *clean_filter_cols,
    *candidate_feature_cols,
]))

training_df = features_df.select(*[c for c in selected_cols if c in features_df.columns])
if TRAIN_START_DATE:
    training_df = training_df.where(F.col("feature_date") >= F.to_date(F.lit(TRAIN_START_DATE)))
if TRAIN_END_DATE:
    training_df = training_df.where(F.col("feature_date") <= F.to_date(F.lit(TRAIN_END_DATE)))
if TRAIN_ONLY_RULE_CLEAN_RECORDS and "has_rule_failure_flag" in training_df.columns:
    strict_clean_condition = F.col("has_rule_failure_flag") == 0
    if {"total_rule_fail_count", "rule_medium_producer_ts_bad_format_flag"}.issubset(set(training_df.columns)):
        bronze_clean_condition = (
            (F.col("has_rule_failure_flag") == 0)
            | ((F.col("total_rule_fail_count") - F.col("rule_medium_producer_ts_bad_format_flag")) == 0)
        )
    elif {"critical_rule_fail_count", "high_rule_fail_count"}.issubset(set(training_df.columns)):
        bronze_clean_condition = (F.col("critical_rule_fail_count") == 0) & (F.col("high_rule_fail_count") == 0)
    else:
        bronze_clean_condition = strict_clean_condition

    training_df = training_df.where(
        F.when(F.col("layer") == "bronze", bronze_clean_condition)
         .otherwise(strict_clean_condition)
    )

if MAX_TRAIN_RECORDS_PER_LAYER > 0:
    sample_window = Window.partitionBy("layer").orderBy(F.rand(TRAIN_SAMPLE_SEED))
    training_df = (
        training_df
        .withColumn("_sample_rank", F.row_number().over(sample_window))
        .where(F.col("_sample_rank") <= F.lit(MAX_TRAIN_RECORDS_PER_LAYER))
        .drop("_sample_rank")
    )

layer_counts_df = training_df.groupBy("layer").count().orderBy("layer")
display(layer_counts_df)

pdf = training_df.orderBy("layer", "event_ts", "record_id").toPandas()
if pdf.empty:
    raise ValueError(
        "No record-level training features found. Run aiops_build_record_features first, "
        "or relax train_start_date/train_end_date/train_only_rule_clean_records."
    )

for c in candidate_feature_cols:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

print(json.dumps({
    "input_record_count_collected_to_driver": int(len(pdf)),
    "max_train_records_per_layer": MAX_TRAIN_RECORDS_PER_LAYER,
    "layers": sorted([str(x) for x in pdf["layer"].dropna().unique().tolist()]),
    "candidate_feature_count": len(candidate_feature_cols),
    "candidate_features": candidate_feature_cols,
}, indent=2))


In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        bottleneck_dim = max(2, hidden_dim // 2)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def serialize_model(model: nn.Module) -> str:
    buffer = BytesIO()
    torch.save(model.state_dict(), buffer)
    return base64.b64encode(buffer.getvalue()).decode("ascii")


def choose_feature_columns(layer_pdf: pd.DataFrame) -> list[str]:
    feature_cols = []
    for c in candidate_feature_cols:
        values = layer_pdf[c].astype("float32")
        if float(values.abs().sum()) == 0.0:
            continue
        if float(values.std()) == 0.0:
            continue
        feature_cols.append(c)
    return feature_cols

In [ ]:
artifact_rows = []
summary_rows = []
skipped_rows = []

for layer, layer_pdf in pdf.groupby("layer"):
    layer_pdf = layer_pdf.sort_values(["event_ts", "record_id"]).copy()
    feature_cols = choose_feature_columns(layer_pdf)

    if len(layer_pdf) < MIN_TRAIN_RECORDS:
        skipped_rows.append({
            "layer": str(layer),
            "reason": f"not_enough_train_records_{len(layer_pdf)}",
            "train_record_count": int(len(layer_pdf)),
            "candidate_feature_count": int(len(feature_cols)),
        })
        print(f"Skipping {layer}: only {len(layer_pdf)} training records.")
        continue

    if not feature_cols:
        skipped_rows.append({
            "layer": str(layer),
            "reason": "no_varying_non_zero_features",
            "train_record_count": int(len(layer_pdf)),
            "candidate_feature_count": 0,
        })
        print(f"Skipping {layer}: no varying non-zero feature columns.")
        continue

    X_raw = layer_pdf[feature_cols].astype("float32").to_numpy()
    mean = X_raw.mean(axis=0)
    std = X_raw.std(axis=0)
    std[std == 0] = 1.0
    X = (X_raw - mean) / std

    torch.manual_seed(42)
    model = AutoEncoder(input_dim=len(feature_cols), hidden_dim=HIDDEN_DIM)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()
    X_tensor = torch.tensor(X, dtype=torch.float32)

    loss_history = []
    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad()
        reconstructed = model(X_tensor)
        loss = loss_fn(reconstructed, X_tensor)
        loss.backward()
        optimizer.step()
        loss_history.append(float(loss.detach().numpy()))

    model.eval()
    with torch.no_grad():
        reconstructed = model(X_tensor).numpy()

    train_errors = np.mean((X - reconstructed) ** 2, axis=1)
    threshold = float(np.quantile(train_errors, ANOMALY_PERCENTILE))

    artifact_rows.append({
        "model_version": MODEL_VERSION,
        "model_name": "pytorch_record_autoencoder",
        "granularity": "record",
        "layer": str(layer),
        "source_feature_table": qname(FEATURE_TABLE),
        "feature_columns_json": json.dumps(feature_cols),
        "mean_json": json.dumps(mean.astype(float).tolist()),
        "std_json": json.dumps(std.astype(float).tolist()),
        "hidden_dim": int(HIDDEN_DIM),
        "epochs": int(EPOCHS),
        "anomaly_percentile": float(ANOMALY_PERCENTILE),
        "anomaly_threshold": threshold,
        "model_state_b64": serialize_model(model),
    })
    summary_rows.append({
        "model_version": MODEL_VERSION,
        "model_name": "pytorch_record_autoencoder",
        "granularity": "record",
        "layer": str(layer),
        "source_feature_table": qname(FEATURE_TABLE),
        "feature_count": int(len(feature_cols)),
        "train_record_count": int(len(layer_pdf)),
        "training_start_event_ts": str(layer_pdf["event_ts"].min()),
        "training_end_event_ts": str(layer_pdf["event_ts"].max()),
        "training_start_feature_date": str(layer_pdf["feature_date"].min()),
        "training_end_feature_date": str(layer_pdf["feature_date"].max()),
        "train_only_rule_clean_records": bool(TRAIN_ONLY_RULE_CLEAN_RECORDS),
        "include_rule_flag_features": bool(INCLUDE_RULE_FLAG_FEATURES),
        "epochs": int(EPOCHS),
        "hidden_dim": int(HIDDEN_DIM),
        "final_training_loss": float(loss_history[-1]),
        "anomaly_percentile": float(ANOMALY_PERCENTILE),
        "anomaly_threshold": threshold,
        "min_train_error": float(train_errors.min()),
        "max_train_error": float(train_errors.max()),
        "avg_train_error": float(train_errors.mean()),
    })

if not artifact_rows:
    raise ValueError(f"No layer models were trained. Skipped layers: {skipped_rows}")

artifact_df = spark.createDataFrame(artifact_rows).withColumn("trained_at", F.current_timestamp())
summary_df = spark.createDataFrame(summary_rows).withColumn("trained_at", F.current_timestamp())

artifact_path = f"{MODEL_BASE_PATH.rstrip('/')}/{MODEL_ARTIFACTS_TABLE}"
summary_path = f"{MODEL_BASE_PATH.rstrip('/')}/{TRAINING_SUMMARY_TABLE}"

(
    artifact_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("model_version", "layer")
    .save(artifact_path)
)

(
    summary_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("model_version", "layer")
    .save(summary_path)
)

spark.sql(f"CREATE TABLE IF NOT EXISTS {qname(MODEL_ARTIFACTS_TABLE)} USING DELTA LOCATION '{artifact_path}'")
spark.sql(f"CREATE TABLE IF NOT EXISTS {qname(TRAINING_SUMMARY_TABLE)} USING DELTA LOCATION '{summary_path}'")
spark.sql(f"REFRESH TABLE {qname(MODEL_ARTIFACTS_TABLE)}")
spark.sql(f"REFRESH TABLE {qname(TRAINING_SUMMARY_TABLE)}")

if skipped_rows:
    display(spark.createDataFrame(skipped_rows))

display(summary_df.orderBy("layer"))